# Qwen-3.5-2B benchmarking.
- Dataset: MME-RealWorld
- Inference: vLLM
- Model Weights: HuggingFace

### Setup

Need to connect to `Google Drive` to avoid installing dependencies and dataset/weights each run, connect to `HuggingFace` in order to pull model weights, initialize environment variables.

In [ ]:
# Choose whichever should be tested.
BENCHMARK_MODELS = {
    "qwen3.5-2b": True,
    "qwen3.5-2b-awq": True
}

# If 1, any uncaught exception in any cell below disconnects this Colab
# runtime (`runtime.unassign()`) instead of leaving a GPU-attached runtime
# idle/billing after an unattended run dies partway through. Set to 0 while
# iterating interactively, so a mistake you're actively debugging doesn't
# also kill the runtime out from under you. See the cell right below this
# one for how it's wired up.
BREAK_ON_ERR = 1

# NOTE: this is about the number of GPU *devices*, not GPU cores -- a
# single Colab GPU (e.g. a T4) still has thousands of CUDA cores; that
# count is irrelevant here.
# If 1, the inference cell's GPU memory profiling (NVML, peak/unload-residual
# in memory_gb) runs -- it assumes exactly one GPU at device index 0, which is
# what every Colab accelerator runtime actually gives you. Set to 0 on a
# multi-GPU runtime (not something Colab offers, but relevant if this
# notebook ever runs elsewhere): hardcoding index 0 there would silently
# report only one GPU's memory as if it were the whole picture, so this
# skips that measurement entirely (memory_gb comes back null) instead of
# reporting a wrong number.
SINGLE_GPU_DEVICE = 1

In [2]:
#* Halt-and-disconnect machinery, registered before any Drive/network/GPU
#* cell below runs, so BREAK_ON_ERR (set above) covers the whole notebook,
#* not just the inference cells.
from google.colab import runtime as _runtime


class StopCellExecution(Exception):
    """Raise this to deliberately stop a cell early (e.g. a loud, expected
    skip) without an ugly traceback -- NOT a real failure, so the handler
    below never disconnects the runtime for it."""

    def _render_traceback_(self):  # hides the traceback for this one
        return []


def _break_on_err(shell, etype, evalue, tb, tb_offset=None):
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    if BREAK_ON_ERR and not issubclass(etype, StopCellExecution):
        print("\nBREAK_ON_ERR is set -- disconnecting the runtime.")
        _runtime.unassign()


# Registered for Exception, not BaseException -- KeyboardInterrupt/SystemExit
# (a manual stop, not a failure) intentionally fall through this handler
# unaffected.
get_ipython().set_custom_exc((Exception,), _break_on_err)

In [3]:
# set up hugging face api
import getpass, os
# take api token from secret input
if os.getenv("HF_TOKEN") is not None:
    os.environ["HUGGINGFACE_API_KEY"] = os.environ["HF_TOKEN"]
elif os.getenv("HUGGINGFACE_API_KEY") is not None:
    os.environ["HF_TOKEN"] = os.environ["HUGGINGFACE_API_KEY"]
else:
    os.environ["HUGGINGFACE_API_KEY"] = os.environ["HF_TOKEN"] = getpass.getpass("HF token (input hidden): ")

In [4]:
#* connect to google drive (storage)
from google.colab import drive

# aim: store model weights and dataset in drive
# force_remount ensures retries in case of a connection error
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
#* set up environment variables
#! Should be done at the beginning.
import os

# Google Drive directory, where everything is going to be stored -- weights,
# dataset, and extracted images persist here across sessions so they only
# ever get downloaded/extracted once, not re-fetched every fresh Colab
# runtime. The parts of the pipeline that are Drive-I/O-sensitive (many small
# files) stage a local working copy from here each session instead of
# reading Drive directly -- see the image-staging and weight-download cells.
PROJECT_DIR = os.environ["PROJECT_DIR"] = "/content/drive/MyDrive/qwen3.5-quant-bench"
os.makedirs(PROJECT_DIR, exist_ok=True)

# subdirectory for huggingface cache
HF_HOME = os.environ["HF_HOME"] = os.path.join(PROJECT_DIR, "hf_cache")

# dataset repo id
DS_REPO_ID = "yifanzhang114/MME-RealWorld"

# Repo id + vLLM `quantization` kwarg per benchmarked variant.
# quantization=None means full precision (fp16).
MODEL_CONFIGS = {
    "qwen3.5-2b": {
        "repo_id": "Qwen/Qwen3.5-2B",
        "quantization": None,
    },
    "qwen3.5-2b-awq": {
        "repo_id": "QuantTrio/Qwen3.5-2B-AWQ",
        "quantization": "awq",
    }
}

# vLLM's torch.compile cache (the compiled Triton/CUDA kernels that feed
# CUDA graph capture -- the graphs themselves are cheap to re-capture fresh
# every process start; it's the *compilation* behind them that's slow and
# actually reusable across runs) defaults to ~/.cache/vllm. Point it at local
# disk explicitly via VLLM_CACHE_ROOT rather than relying on that implicit
# default -- the cache is many small files written/read throughout engine
# startup, exactly the access pattern Drive's FUSE mount handles worst (see
# the image-staging cell's notes) -- and round-trip the whole directory
# to/from a single Drive-hosted tarball once per model pass instead (see the
# cache load/save cells around the inference loop below), so Drive only ever
# sees one big sequential file.
VLLM_CACHE_LOCAL = "/content/.cache"
os.environ["VLLM_CACHE_ROOT"] = os.path.join(VLLM_CACHE_LOCAL, "vllm")
VLLM_CACHE_STORE = os.path.join(PROJECT_DIR, "vllm_cache.tar")

# Rust-based accelerated downloader for HF Hub transfers (needs the hf_transfer
# package, installed below) -- meaningfully faster than the default downloader
# for large weight files.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [6]:
%%bash
# Without this, a failed command partway through (e.g. a flaky vllm
# install) doesn't stop the script -- bash just keeps going and the
# cell's exit status ends up being whatever the *last* command returned,
# silently masking the real failure until `import vllm` breaks confusingly
# cells later. set -e aborts on the first failing command instead, so the
# error surfaces here (and BREAK_ON_ERR, above, can actually see it).
set -e
pip install uv # faster than python venv
# Install straight into the Colab kernel's own Python environment with
# --system, rather than `uv venv` + `source .venv/bin/activate`: a %%bash
# cell runs in its own throwaway subprocess, so an activated venv there never
# propagates to the notebook's actual Python kernel process (a separate,
# already-running process using /usr/local/lib/python3.13/dist-packages).
# Anything installed into an isolated .venv here would be invisible to every
# `import vllm`/`import pynvml`/etc. in the cells below -- and uv venv also
# errors on a rerun ("A virtual environment already exists"), unlike --system
# installs, which are naturally idempotent.
uv pip install --system vllm --torch-backend=auto
# vLLM raises an error if torchaudio is installed and CUDA version is different
# But I don't need torchaudio anyway.
uv pip uninstall --system torchaudio
# [hf_transfer] extra pulls in the Rust-based accelerated downloader used by
# HF_HUB_ENABLE_HF_TRANSFER above.
uv pip install --system "huggingface_hub[hf_transfer]"
# GPU memory is read via NVML (nvidia-ml-py) in the inference cell, not
# torch.cuda.* -- vLLM's actual model execution runs in a separate spawned
# subprocess, so torch.cuda.* in this process can't see that memory.
# nvidia-ml-py comes in transitively via vLLM already, but pinning it
# explicitly here means it doesn't silently disappear if a future vLLM
# release drops or changes that transitive dependency.
uv pip install --system nvidia-ml-py
# Dataset evaluation code. Guarded so re-running this cell mid-session doesn't
# error on an already-cloned directory (this lives under /content, not Drive,
# so it does need to be re-cloned every fresh Colab runtime).
if [ ! -d /content/MME-RealWorld ]; then
  git clone https://github.com/MME-Benchmarks/MME-RealWorld.git /content/MME-RealWorld
fi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 118.0 MB/s eta 0:00:00


Using Python 3.13.15 environment at: /usr
Resolved 197 packages in 2.11s
Prepared 100 packages in 35.62s
Uninstalled 15 packages in 633ms
Installed 100 packages in 281ms
 + agent-detector==2.0.0
 + anthropic==1.7.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.9
 + cbor2==6.1.4
 + compressed-tensors==0.17.0
 - cuda-bindings==12.9.7
 + cuda-bindings==13.4.2
 - cuda-core==0.3.2
 + cuda-core==1.2.0
 - cuda-python==12.9.7
 + cuda-python==13.4.1
 + cuda-tile==1.6.0
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.2.1
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dnspython==2.8.0
 + email-validator==2.3.0
 - fastapi==0.141.1
 + fastapi==0.136.3
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.26.0
 + fastar==0.12.0
 + fastsafetensors==0.4.0
 + flashinfer-python==0.6.18
 + humming-kernels==0.1.12
 + ijson==3.5.1
 + instanttensor==0.2.0
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.7.6
 - llvmlite==0.44.0
 + llvmlite==0.47.0
 + lm-format-enforcer=

In [7]:
#* download model weights
import shutil
import time
from concurrent.futures import ThreadPoolExecutor

from huggingface_hub import snapshot_download

# Model weight files are few and large (a handful of safetensors shards) --
# a very different Drive access pattern from the many-small-image problem
# elsewhere in this notebook, and bulk sequential reads are Drive's best
# case, not its worst. Still staged to local disk before vLLM loads them,
# though: safetensors loading commonly memory-maps its files, and mmap'd
# access over a network-backed FUSE mount can turn into scattered small
# reads depending on access order, which would reintroduce the same failure
# mode staging fixed for images. Cheap precaution given the retry-copy
# pattern already exists for exactly this.
LOCAL_WEIGHTS_DIR = "/content/model_weights"

def _stage_file(src: str, dst: str, retries: int = 3, delay: float = 1.0) -> None:
    # Copy to a temp name and rename into place atomically, rather than
    # writing `dst` directly -- otherwise a copy interrupted partway (kernel
    # restart, disconnect) leaves a truncated file under the final filename,
    # which a later run's `os.path.exists(dst)` check would treat as
    # already-staged and never re-copy. For images specifically this is a
    # silent-corruption risk, not just a "redo the copy" one: a truncated
    # JPEG often still decodes via PIL with no error, just visible
    # corruption, silently feeding a broken image into the model instead of
    # failing loudly. Shared by every staging call site in this notebook
    # (model weights here, sampled images further down) -- keep it that way
    # rather than forking a near-identical copy for the next thing that
    # needs staging.
    tmp_dst = dst + ".part"
    for attempt in range(retries):
        try:
            shutil.copyfile(src, tmp_dst)
            os.replace(tmp_dst, dst)
            return
        except OSError:
            if attempt == retries - 1:
                raise
            time.sleep(delay)

def _download_model_weights(key: str, repo_id: str) -> tuple[str, str]:
    drive_dir = snapshot_download(repo_id=repo_id, cache_dir=HF_HOME)
    local_dir = os.path.join(LOCAL_WEIGHTS_DIR, key)
    os.makedirs(local_dir, exist_ok=True)
    for name in os.listdir(drive_dir):
        src = os.path.join(drive_dir, name)
        if not os.path.isfile(src):  # snapshot dirs are flat in practice
            continue
        dst = os.path.join(local_dir, name)
        if not os.path.exists(dst):
            _stage_file(src, dst)
    return key, local_dir

to_download = {
    key: cfg["repo_id"]
    for key, cfg in MODEL_CONFIGS.items()
    if BENCHMARK_MODELS.get(key) and cfg["repo_id"]
}
for key in BENCHMARK_MODELS:
    if BENCHMARK_MODELS[key] and key not in to_download:
        print(f"Skipping {key}: no repo_id configured in MODEL_CONFIGS.")

# Every model's weights are an independent HF download -- fetch them
# concurrently instead of waiting on each snapshot_download in turn.
with ThreadPoolExecutor(max_workers=max(1, len(to_download))) as pool:
    for key, weights_path in pool.map(lambda kv: _download_model_weights(*kv), to_download.items()):
        MODEL_CONFIGS[key]["weights_path"] = weights_path
        print(f"Finished downloading {key}: {weights_path}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Finished downloading qwen3.5-2b: /content/model_weights/qwen3.5-2b
Finished downloading qwen3.5-2b-awq: /content/model_weights/qwen3.5-2b-awq


In [8]:
#* download dataset
from huggingface_hub import snapshot_download

# download dataset
dataset_path = snapshot_download(
    repo_id=DS_REPO_ID,
    repo_type="dataset",
    cache_dir=HF_HOME
)

SNAPSHOT_ROOT = dataset_path

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

In [9]:
#* check if there's any broken archive
import glob

def check_all_targets(snapshot_root: str) -> list[str]:
    broken = []
    for f in glob.glob(os.path.join(snapshot_root, "*")):
        if os.path.islink(f) and not os.path.exists(f):
            broken.append(f)
    return broken

broken = check_all_targets(SNAPSHOT_ROOT)
for b in broken:
    print("BROKEN:", os.path.basename(b))
print(f"\n{len(broken)} broken symlinks found out of total files")


0 broken symlinks found out of total files


In [ ]:
#* download broken archives again
from huggingface_hub import hf_hub_download

for b in broken:
    filename = os.path.basename(b)
    print(f"Re-downloading {filename}...")
    hf_hub_download(
        repo_id=DS_REPO_ID,
        repo_type="dataset",
        filename=b,
        cache_dir=HF_HOME,
        force_download=True,
    )

In [11]:
if check_all_targets(SNAPSHOT_ROOT):
    raise RuntimeError("Broken symlinks still exist")

In [12]:
import base64
import os

# vLLM accepts images as base64 data URLs (image_url content type), not just
# PIL objects -- reading raw bytes and base64-encoding them skips a PIL
# decode on our side entirely (vLLM decodes server-side), and the base64
# string is far lighter to hold in memory than a decoded RGB array would be.
_MIME_BY_EXT = {".jpg": "jpeg", ".jpeg": "jpeg", ".png": "png", ".webp": "webp"}

def image_to_data_url(image_path: str) -> str:
  ext = os.path.splitext(image_path)[1].lower()
  mime = _MIME_BY_EXT.get(ext, "jpeg")
  with open(image_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")
  return f"data:image/{mime};base64,{b64}"

In [13]:
import os, glob

# locate question list json
json_candidates = glob.glob(os.path.join(SNAPSHOT_ROOT, "**", "*.json"), recursive=True)
assert json_candidates, "No JSON file found in snapshot — inspect the printed listing above."

print(f"Discovered potential JSON files: {json_candidates}")

# ignore chinese and look for english
QUESTIONS_FILE = [candidate for candidate in json_candidates if "MME" in os.path.basename(candidate) and not "CN" in candidate][0]

print("Using questions file:", QUESTIONS_FILE)

Discovered potential JSON files: ['/content/drive/MyDrive/qwen3.5-quant-bench/hf_cache/datasets--yifanzhang114--MME-RealWorld/snapshots/741cb8831ac86085bd54f678d13ca193e2334114/MME_RealWorld_CN.json', '/content/drive/MyDrive/qwen3.5-quant-bench/hf_cache/datasets--yifanzhang114--MME-RealWorld/snapshots/741cb8831ac86085bd54f678d13ca193e2334114/MME_RealWorld.json']
Using questions file: /content/drive/MyDrive/qwen3.5-quant-bench/hf_cache/datasets--yifanzhang114--MME-RealWorld/snapshots/741cb8831ac86085bd54f678d13ca193e2334114/MME_RealWorld.json


In [14]:
PROMPT_SUFFIX = (
    "Select the best answer to the above multiple-choice question based on the image. "
    "Respond with only the letter (A, B, C, D, or E) of the correct option.\n"
    "The best answer is:"
)

from typing import Dict

def build_chat(item: Dict) -> Dict:
  """Take question item from dataset and turn into prompt list."""
  choices_text = "The choices are listed below:\n" + "\n".join(item["Answer choices"])
  text = f"{item['Text']}\n{choices_text}\n{PROMPT_SUFFIX}"
  # IMAGE_CACHE (loaded from Drive, see the cell right before the inference
  # loop below) is checked first -- a hit skips the local read + base64
  # encode entirely. On a miss, encode once and store it, so every later
  # call for this same image -- the next model pass, or a future session
  # that reloads the cache from Drive -- hits the cache instead of
  # re-encoding. This function runs inside run_inference_pass's background
  # thread pools, so filling a miss happens concurrently with GPU inference
  # on whatever window is currently running, not as a separate blocking step.
  basename = os.path.basename(item["Image"])
  data_url = IMAGE_CACHE.get(basename)
  if data_url is None:
    image_path = os.path.join(EXTRACT_DIR, basename)
    data_url = IMAGE_CACHE[basename] = image_to_data_url(image_path)
  return [
      {
          "role": "user",
          "content": [
              {"type": "image_url", "image_url": {"url": data_url}},
              {"type": "text", "text": text},
          ],
      }
  ]


## Inference

In [15]:
# construct sample dataset
# take 10% of each task
import os, json

SAMPLE_QUESTIONS_FILE = os.path.join(PROJECT_DIR, "sample_questions.json")

if os.path.exists(SAMPLE_QUESTIONS_FILE):
  with open(SAMPLE_QUESTIONS_FILE, "r") as f:
    questions = json.load(f)
else:
  import random

  # Fixed seed: the sample itself is already cached and reused across
  # all three model passes within a run (that's the real 'comparable'
  # guarantee) -- seeding on top of that just makes a freshly regenerated
  # sample (e.g. after deleting the cache or changing DS_COMPR_RATE)
  # reproducible run-to-run too, instead of silently comparing different
  # 10% subsets if the cache is ever rebuilt.
  random.seed(0)

  # Only load the full question set when we actually need to sample from it --
  # parsing all ~23,600 questions is wasted work on every run that already
  # has a cached sample.
  with open(QUESTIONS_FILE, "r") as f:
    all_questions = json.load(f)
  print(f"Loaded {len(all_questions)} questions")

  DS_COMPR_RATE = 0.1

  reas = [q for q in all_questions if q.get('Task') == 'Reasoning']
  perc = [q for q in all_questions if q.get('Task') == 'Perception']

  reas = random.sample(reas, int(DS_COMPR_RATE * len(reas)))
  perc = random.sample(perc, int(DS_COMPR_RATE * len(perc)))

  questions = reas + perc

  with open(SAMPLE_QUESTIONS_FILE, "w") as f:
    json.dump(questions, f, indent=2)

print(f"Using {len(questions)} sampled questions")

Using 2360 sampled questions


In [16]:
import gc
import json
import threading
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pynvml
import torch
from vllm import LLM, SamplingParams
from vllm.distributed.parallel_state import destroy_model_parallel

# Chunk size doubles as vLLM's concurrent scheduling window (max_num_seqs) and
# the incremental-checkpoint boundary, so each chunk is sized to actually
# saturate the scheduler (continuous batching), not an arbitrary number
# unrelated to how many requests vLLM can run at once.
MAX_NUM_SEQS = 64
BATCH_SIZE = MAX_NUM_SEQS
# Batch sizes vLLM pre-captures a CUDA graph for. Covers the partial last chunk too.
CUDAGRAPH_SIZES = [1, 2, 4, 8, 16, 32, 64]

# Chat payloads are built a few inference chunks at a time, not for the whole
# sample up front: images come from the local staged copy, so building a
# small window at a time keeps only that window's worth of data resident
# instead of scaling memory with sample size, at the cost of redoing the
# (now-cheap, local, base64) read/encode per model pass instead of once for
# all three. The next window is also prefetched on a background thread while
# the current window's GPU inference runs (below), since that inference
# takes meaningfully longer than reading+encoding a window's worth of local
# images -- this hides most of the build time behind GPU time instead of
# paying for it serially between every window.
GROUP_BATCHES = 3
CHAT_GROUP_SIZE = GROUP_BATCHES * BATCH_SIZE

# GPU memory is queried through NVML (via pynvml, already installed as a
# vLLM dependency), not torch.cuda.* -- vLLM's engine actually runs the model
# in a separate spawned subprocess (see the "must use spawn... CUDA is
# initialized" log line during engine startup), so torch.cuda.* calls made
# from this process only ever see this process's own near-empty CUDA
# context, never the child process where the real GPU memory is used. NVML
# queries the GPU device itself, which is correct regardless of which
# process is using it.
#
# Hardcoding device index 0 below only makes sense on a single-GPU runtime,
# so it's gated behind SINGLE_GPU_DEVICE (config cell) rather than assumed
# unconditionally -- on a multi-GPU box this would otherwise silently report
# only one GPU's memory as if it were the whole picture.
if SINGLE_GPU_DEVICE:
    pynvml.nvmlInit()
    _GPU_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
else:
    _GPU_HANDLE = None
    print(
        "SINGLE_GPU_DEVICE is not set -- skipping NVML GPU memory profiling "
        "(memory_gb.peak_allocated/unload_residual will be null)."
    )


def _gpu_mem_used_bytes() -> int | None:
    if _GPU_HANDLE is None:
        return None
    return pynvml.nvmlDeviceGetMemoryInfo(_GPU_HANDLE).used


def _percentiles(values: list[float]) -> dict:
    if not values:
        return {"p50": None, "p90": None, "p99": None}
    arr = np.array(values)
    return {
        "p50": float(np.percentile(arr, 50)),
        "p90": float(np.percentile(arr, 90)),
        "p99": float(np.percentile(arr, 99)),
    }


def _stat_block(values: list[float]) -> dict:
    """avg + p50/p90/p99 for one metric's raw per-request values -- the
    shape shared by latency/ttft/tpot/queue_time/prefill_time/decode_time
    below. Always computed from the raw per-request list, never from
    already-averaged per-batch numbers, which would silently collapse the
    tail these percentiles exist to show."""
    return {"avg": float(np.mean(values)) if values else None, **_percentiles(values)}


def _request_metrics(output) -> dict | None:
    """Per-request timing off vLLM's own request stats (RequestOutput.metrics).
    Populated only when disable_log_stats=False -- the offline LLM class
    defaults this to True (disabled), which is why every metric came back
    null the first time this ran. Field names here match vLLM's current
    RequestStateStats (vllm/v1/metrics/stats.py), restored after v1
    originally dropped per-request metrics entirely (see vllm-project/vllm
    PR #24947) -- NOT the older RequestMetrics naming (first_token_time,
    finished_time) that older docs/examples may still show. That restored
    API is explicitly flagged by vLLM's own maintainers as not yet stable,
    so every field is read defensively (getattr, not direct attribute
    access) -- a future vLLM release renaming these again should degrade
    that one field to None rather than crashing the whole run.

    RequestStateStats also carries num_preemptions and is_corrupted, which
    aren't timings but matter for benchmark validity: a preempted request's
    latency includes time recomputing work that was evicted for another
    request's KV cache, and a corrupted request's output shouldn't be
    trusted for the accuracy eval either. Both are surfaced per-request here
    so run_inference_pass can aggregate and warn on them.

    finish_reason comes from CompletionOutput (vllm/outputs.py), not
    RequestStateStats -- a stable, non-experimental field, so it's read
    directly rather than defensively. Worth tracking specifically because
    max_tokens=512 below is a hard cap: a request that hits it
    (finish_reason == "length") was truncated before the model necessarily
    finished its answer, which both skews that request's latency/TPOT
    (measuring against an artificial cutoff, not the model's natural stop)
    and risks feeding the accuracy eval an answer that got cut off
    mid-generation."""
    m = output.metrics
    if m is None:
        return None
    first_tok = getattr(m, "first_token_ts", None)
    last_tok = getattr(m, "last_token_ts", None)
    if first_tok is None or last_tok is None:
        return None
    queued_ts = getattr(m, "queued_ts", None)
    scheduled_ts = getattr(m, "scheduled_ts", None)
    num_preemptions = getattr(m, "num_preemptions", None)
    is_corrupted = getattr(m, "is_corrupted", None)
    completion = output.outputs[0]
    ntoks = len(completion.token_ids)
    # Clock domains matter here: arrival_time is wall-clock epoch seconds
    # (time.time()), while queued_ts/scheduled_ts/first_token_ts/last_token_ts
    # are the engine core's monotonic timestamps. Subtracting arrival_time from
    # them yields nonsense (~ -1.79e9 s in a real run). All three request-level
    # durations are therefore measured from queued_ts, which shares a clock
    # with the other timestamps; this excludes only the tiny frontend time
    # before the request reaches the engine queue.
    start = queued_ts if queued_ts else None  # dataclass default 0.0 == unset
    return {
        "latency": last_tok - start if start is not None else None,
        "ttft": first_tok - start if start is not None else None,
        "tpot": (last_tok - first_tok) / (ntoks - 1) if ntoks > 1 else None,
        # Mirrors vLLM's own internal breakdown of where request time goes
        # (see IterationStats in vllm/v1/metrics/stats.py): time waiting to
        # be scheduled, then scheduled-to-first-token (prefill), then
        # first-to-last-token (decode). Each is independently None (not
        # just missing from the dict) if its underlying timestamp isn't
        # populated on the installed vLLM version.
        "queue_time": (
            scheduled_ts - queued_ts
            if scheduled_ts is not None and queued_ts is not None
            else None
        ),
        "prefill_time": (
            first_tok - scheduled_ts if scheduled_ts is not None else None
        ),
        "decode_time": last_tok - first_tok,
        "num_tokens": ntoks,
        "num_preemptions": num_preemptions,
        "is_corrupted": bool(is_corrupted) if is_corrupted is not None else None,
        "finish_reason": completion.finish_reason,
    }


def run_inference_pass(
    model_weights_cache: str,
    results_json: str,
    questions: list[dict],
    quantization: str | None = None,
) -> dict:
    llm_kwargs = dict(
        model=model_weights_cache,
        # T4 (this notebook's Colab GPU) has no native bf16 tensor cores --
        # float16 is the correct dtype here, not just a default choice.
        dtype="float16",
        # Kept the same across every quantization method under test on purpose:
        # letting a smaller quantized model claim more KV-cache headroom would
        # improve its throughput for a reason unrelated to quantization itself,
        # which would make the comparison unfair.
        gpu_memory_utilization=0.85,
        # MME-RealWorld images vary widely in resolution, and vision
        # tokenization scales with it -- at least one image in this sample
        # needs 16000+ tokens once encoded, well past a "short prompt"
        # assumption. Left with real headroom above that observed floor
        # (not tuned to the exact minimum) so an even larger outlier image
        # doesn't hit the same wall; still far below this model's native
        # 262144-token max, since leaving max_model_len unset sizes the KV
        # cache for that full context across every one of max_num_seqs
        # concurrent sequences, which can fail engine startup outright.
        max_model_len=32768,
        max_num_seqs=MAX_NUM_SEQS,
        trust_remote_code=True,
        # Disabled: this model uses a Mamba-hybrid architecture, and vLLM
        # switches into a special Mamba cache "align" mode specifically
        # because this is on -- engine startup fails immediately after that
        # log line. The win from prefix caching here was always modest
        # (only the shared chat-template preamble benefits; the actual
        # image+question content differs per request), not worth trading
        # for a working engine. Revisit once vLLM's hybrid-model prefix
        # caching support matures.
        enable_prefix_caching=False,
        # Explicitly enabled: the offline LLM class defaults this to True
        # (disabled), which silently means every RequestOutput.metrics comes
        # back None -- this is the actual switch that turns per-request
        # timing on, not something inferred from other settings.
        disable_log_stats=False,
        compilation_config={"cudagraph_capture_sizes": CUDAGRAPH_SIZES},
    )
    if quantization:
        llm_kwargs["quantization"] = quantization
        # NOTE: vLLM auto-upgrades "awq" to the Marlin-kernel backend on
        # Ampere+ GPUs. T4 is Turing (sm_75), which Marlin does not support, so
        # expect vLLM to fall back to a slower (but correct) kernel here --
        # check the engine startup log for which kernel actually got selected,
        # since that dominates quantized throughput far more than anything else
        # tunable in this cell.

    mem_before = _gpu_mem_used_bytes()

    # vLLM pre-allocates its KV cache pool up to gpu_memory_utilization right
    # at engine startup, so usage is close to flat once loaded rather than
    # spiking sharply during inference -- but polling on a background thread
    # for the whole call (not just sampling once after load) catches any
    # real spike (e.g. activation memory on an unusually large batch) that a
    # single snapshot would miss, at negligible cost. Skipped entirely when
    # SINGLE_GPU_DEVICE is unset -- _GPU_HANDLE is None then, so there's nothing
    # for a poll loop to read.
    mem_samples = [mem_before] if SINGLE_GPU_DEVICE else []
    stop_polling = threading.Event()
    poll_thread = None

    def _poll_gpu_memory() -> None:
        while not stop_polling.is_set():
            mem_samples.append(_gpu_mem_used_bytes())
            stop_polling.wait(0.2)

    if SINGLE_GPU_DEVICE:
        poll_thread = threading.Thread(target=_poll_gpu_memory, daemon=True)
        poll_thread.start()

    load_start = time.perf_counter()
    llm = LLM(**llm_kwargs)
    model_load_seconds = time.perf_counter() - load_start
    sampling_params = SamplingParams(temperature=0.0, max_tokens=512)

    results = []
    request_metrics = []

    try:
        wall_start = time.perf_counter()

        group_ranges = [
            (g, min(g + CHAT_GROUP_SIZE, len(questions)))
            for g in range(0, len(questions), CHAT_GROUP_SIZE)
        ]

        # Two pools: img_pool parallelizes the per-item reads within one group's
        # build; prefetch_pool (a single worker) runs one group-build ahead of
        # where inference currently is, so that build overlaps with this group's
        # llm.chat() calls below instead of happening serially before them.
        with ThreadPoolExecutor(max_workers=8) as img_pool, \
             ThreadPoolExecutor(max_workers=1) as prefetch_pool:

            def _build_group(start: int, end: int) -> list:
                return list(img_pool.map(build_chat, questions[start:end]))

            next_future = (
                prefetch_pool.submit(_build_group, *group_ranges[0]) if group_ranges else None
            )
            for idx, (g_start, g_end) in enumerate(group_ranges):
                group_items = questions[g_start:g_end]
                group_chats = next_future.result()

                if idx + 1 < len(group_ranges):
                    next_future = prefetch_pool.submit(_build_group, *group_ranges[idx + 1])

                for i in range(0, len(group_items), BATCH_SIZE):
                    batch_items = group_items[i : i + BATCH_SIZE]
                    batch_chats = group_chats[i : i + BATCH_SIZE]
                    # one llm.chat() call per chunk -- vLLM continuously batches
                    # every request handed to it together, rather than us
                    # serializing on chunk boundaries smaller than the scheduler
                    # could actually run.
                    outputs = llm.chat(batch_chats, sampling_params)
                    for item, output in zip(batch_items, outputs):
                        item_copy = dict(item)
                        item_copy["Output"] = output.outputs[0].text.strip()
                        results.append(item_copy)
                        rm = _request_metrics(output)
                        if rm is not None:
                            request_metrics.append(rm)

                    # write incrementally in case kernel dies
                    with open(results_json, "w") as f:
                        json.dump(results, f)

                    print(f"Processed {g_start + i + len(batch_items)}/{len(questions)}")
        wall_seconds = time.perf_counter() - wall_start
    finally:
        # Runs even if the loop above raised (a bad request, an engine
        # crash mid-run, BREAK_ON_ERR about to fire) -- otherwise a crashed
        # pass leaves this model resident on the GPU, contaminating the
        # next model's mem_before baseline if the notebook is rerun in the
        # same kernel session without a restart.
        stop_polling.set()
        if poll_thread is not None:
            poll_thread.join(timeout=1.0)
        peak_mem = max(mem_samples) if mem_samples else None

        destroy_model_parallel()
        del llm
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

        mem_after_unload = None
        if SINGLE_GPU_DEVICE:
            # The freed reference above only tears down this (near-empty)
            # process's own CUDA context -- the actual GPU memory lives in
            # vLLM's spawned engine subprocess (see the NVML note above), which
            # takes a moment to actually exit after being torn down. Poll until
            # usage stops dropping (bounded so a genuine leak doesn't hang this
            # cell forever) instead of a single snapshot that could catch that
            # process mid-exit and overstate unload_residual.
            mem_after_unload = _gpu_mem_used_bytes()
            for _ in range(50):  # up to ~10s
                time.sleep(0.2)
                reading = _gpu_mem_used_bytes()
                if reading >= mem_after_unload:
                    break
                mem_after_unload = reading

    # If every single request came back with no usable metrics, that's the
    # exact failure mode that motivated this rewrite (disable_log_stats
    # defaulting to on, or a future vLLM release renaming these fields
    # again) -- surface it loudly instead of silently writing another
    # all-null metrics file.
    if results and not request_metrics:
        print(
            f"WARNING: {len(results)} requests completed but none produced "
            f"usable per-request metrics -- latency/TTFT/TPOT will be null "
            f"below. Check disable_log_stats and the RequestStateStats field "
            f"names in _request_metrics against the installed vLLM version."
        )

    latencies = [r["latency"] for r in request_metrics if r["latency"] is not None]
    ttfts = [r["ttft"] for r in request_metrics if r["ttft"] is not None]
    tpots = [r["tpot"] for r in request_metrics if r["tpot"] is not None]
    ntoks = [r["num_tokens"] for r in request_metrics]
    queue_times = [r["queue_time"] for r in request_metrics if r["queue_time"] is not None]
    prefill_times = [r["prefill_time"] for r in request_metrics if r["prefill_time"] is not None]
    decode_times = [r["decode_time"] for r in request_metrics]
    preemption_counts = [
        r["num_preemptions"] for r in request_metrics if r["num_preemptions"] is not None
    ]
    corrupted_flags = [r["is_corrupted"] for r in request_metrics if r["is_corrupted"] is not None]
    truncated_flags = [r["finish_reason"] == "length" for r in request_metrics]

    if any(corrupted_flags):
        print(
            f"WARNING: {sum(corrupted_flags)}/{len(request_metrics)} requests "
            f"were flagged is_corrupted by vLLM -- their outputs (and the "
            f"accuracy eval that consumes them) may not be trustworthy."
        )

    if any(truncated_flags):
        print(
            f"WARNING: {sum(truncated_flags)}/{len(request_metrics)} requests "
            f"hit max_tokens={sampling_params.max_tokens} (finish_reason="
            f"'length') and were truncated -- their latency/TPOT reflect an "
            f"artificial cutoff, not the model's natural stop, and their "
            f"answer may have been cut off before the accuracy eval sees it."
        )

    time_weighted_latency = (
        sum(r["latency"] * r["num_tokens"] for r in request_metrics if r["latency"] is not None)
        / sum(r["num_tokens"] for r in request_metrics if r["latency"] is not None)
        if latencies and sum(r["num_tokens"] for r in request_metrics if r["latency"] is not None) > 0
        else None
    )

    metrics = {
        "num_requests": len(results),
        "model_load_seconds": model_load_seconds,
        "wall_clock_seconds": wall_seconds,
        "latency": {**_stat_block(latencies), "time_weighted_avg": time_weighted_latency},
        "ttft": _stat_block(ttfts),
        "tpot": _stat_block(tpots),
        # Decomposition of latency into where the time actually went (see
        # _request_metrics above) -- e.g. a high queue_time under this
        # notebook's 64-concurrent-sequence load points at scheduler
        # backpressure, not slow per-token generation.
        "queue_time": _stat_block(queue_times),
        "prefill_time": _stat_block(prefill_times),
        "decode_time": _stat_block(decode_times),
        "throughput": {
            "requests_per_second": len(results) / wall_seconds if wall_seconds > 0 else None,
            "output_tokens_per_second": sum(ntoks) / wall_seconds if wall_seconds > 0 else None,
        },
        "preemptions": {
            "total": int(sum(preemption_counts)) if preemption_counts else None,
            "requests_affected": (
                sum(1 for c in preemption_counts if c > 0) if preemption_counts else None
            ),
        },
        "corrupted_requests": sum(corrupted_flags) if corrupted_flags else None,
        # Count of requests that hit max_tokens rather than stopping
        # naturally -- see the WARNING above for why this matters for both
        # latency interpretation and accuracy-eval trustworthiness.
        "truncated_requests": sum(truncated_flags) if request_metrics else None,
        "memory_gb": {
            "peak_allocated": peak_mem / 1e9 if peak_mem is not None else None,
            "unload_residual": (
                (mem_after_unload - mem_before) / 1e9
                if mem_after_unload is not None and mem_before is not None
                else None
            ),
        },
    }
    return {"results": results, "metrics": metrics}


In [17]:
#* load the persistent base64 image cache from Drive, if one exists
import json
import os

# Reading + base64-encoding an image is redone once per model pass (chat
# payloads are rebuilt per pass, see run_inference_pass above) and again on
# every fresh Colab session, even though the same sample produces the exact
# same encoded string every time. Rather than building this cache in a
# separate blocking step before inference starts, build_chat (above) fills
# it lazily on a cache miss -- since it already runs inside
# run_inference_pass's background thread pools, filling a miss happens
# concurrently with GPU inference on whatever window is currently running,
# the same way window-prefetching already overlaps chat-building with
# inference. This cell only loads whatever was already cached from a prior
# run; save_image_cache() (defined here, called from the driving loop below
# after every model pass -- same placement as sync_vllm_cache_to_drive, and
# for the same reason) persists it back to Drive, so a crash partway
# through a multi-model run doesn't throw away the cache-filling work
# already done.
IMAGE_CACHE_FILE = os.path.join(PROJECT_DIR, "base64_image_cache.json")

if os.path.exists(IMAGE_CACHE_FILE):
    with open(IMAGE_CACHE_FILE, "r") as f:
        IMAGE_CACHE = json.load(f)
    print(f"Loaded {len(IMAGE_CACHE)} cached images from {IMAGE_CACHE_FILE}")
else:
    IMAGE_CACHE = {}
    print("No cached images found on Drive yet -- starting with an empty cache.")


def save_image_cache() -> None:
    # Same atomic-write pattern as _stage_file (write to a temp name, then
    # os.replace() into place) -- a write interrupted partway through this
    # (potentially large) file would otherwise leave a truncated,
    # unparseable JSON file under the real filename, which a later run's
    # os.path.exists() check would treat as a complete, valid cache.
    tmp_cache_file = IMAGE_CACHE_FILE + ".part"
    with open(tmp_cache_file, "w") as f:
        json.dump(IMAGE_CACHE, f)
    os.replace(tmp_cache_file, IMAGE_CACHE_FILE)
    print(f"Saved {len(IMAGE_CACHE)} cached images to {IMAGE_CACHE_FILE}")


Loaded 2174 cached images from /content/drive/MyDrive/qwen3.5-quant-bench/base64_image_cache.json


In [18]:
#* make sure every image the sample still needs is on local disk
import glob
import os
import subprocess
from concurrent.futures import ThreadPoolExecutor

# Everything below reads images from local disk (EXTRACT_DIR); nothing
# downstream touches Drive for images. What varies is how the images that
# aren't already covered by IMAGE_CACHE get there, cheapest path first:
#
#   1. IMAGE_CACHE (Drive, loaded above) already has every image the sample
#      needs -> no raw image files are needed at all; skip everything.
#   2. Otherwise stage just the missing ones from PROJECT_DIR/mme_dataset_images
#      (the flattened copy already on Drive) -- a few thousand small reads
#      once, with _stage_file's retries, then the cache makes it a one-off.
#   3. Only if that Drive copy doesn't exist or is incomplete: copy the
#      dataset archives to local disk and extract + flatten them there
#      (tar writing ~23,600 small files straight onto Drive's FUSE mount is
#      the write-side version of the many-small-files problem, so extraction
#      is always local).
EXTRACT_DIR = "/content/mme_dataset_images"
os.makedirs(EXTRACT_DIR, exist_ok=True)
DRIVE_FLAT_DIR = os.path.join(PROJECT_DIR, "mme_dataset_images")
_IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")


def _extract_archives_locally() -> None:
    local_archive_dir = "/content/mme_archives"
    os.makedirs(local_archive_dir, exist_ok=True)
    archive_paths = sorted(
        glob.glob(os.path.join(SNAPSHOT_ROOT, "*.tar.gz"))
        + glob.glob(os.path.join(SNAPSHOT_ROOT, "*.tar.gz.part_*"))
    )
    to_copy = [
        p for p in archive_paths
        if not os.path.exists(os.path.join(local_archive_dir, os.path.basename(p)))
    ]
    # Archives are few and large -- Drive's best case for a bulk copy.
    with ThreadPoolExecutor(max_workers=max(1, len(to_copy))) as pool:
        list(pool.map(
            lambda p: _stage_file(p, os.path.join(local_archive_dir, os.path.basename(p))),
            to_copy,
        ))
    print(f"{len(archive_paths) - len(to_copy)} archives already local, {len(to_copy)} copied.")

    part_files = glob.glob(os.path.join(local_archive_dir, "*.tar.gz.part_*"))
    plain_files = glob.glob(os.path.join(local_archive_dir, "*.tar.gz"))
    # base name -> path, so plain files always win over parts when both exist
    plain_by_base = {f[:-len(".tar.gz")]: f for f in plain_files}
    part_bases = sorted(set(f.rsplit(".tar.gz.part_", 1)[0] for f in part_files))

    to_extract = []  # (archive_path, needs_reassembly, parts)
    handled_bases = set()
    for base in part_bases:
        plain = plain_by_base.get(base)
        if plain and os.path.exists(plain):
            print(f"Using existing {os.path.basename(plain)} (skipping .part_ files for this base)")
            to_extract.append((plain, False, None))
        else:
            to_extract.append((base + ".tar.gz", True, sorted(glob.glob(base + ".tar.gz.part_*"))))
        handled_bases.add(base)
    for base, plain in plain_by_base.items():
        if base not in handled_bases:
            to_extract.append((plain, False, None))

    for archive_name, needs_reassembly, parts in to_extract:
        # A plain archive always exists on disk, so the archive's own
        # existence says nothing about whether it was extracted -- track
        # completion with a marker instead (guards a same-session rerun;
        # local disk is wiped on a fresh runtime anyway).
        marker = archive_name + ".extracted"
        if os.path.exists(marker):
            continue
        if needs_reassembly and not os.path.exists(archive_name):
            print(f"Reassembling {os.path.basename(archive_name)} from {len(parts)} parts...")
            with open(archive_name, "wb") as out_f:
                for p in parts:
                    with open(p, "rb") as in_f:
                        out_f.write(in_f.read())
        print(f"Extracting {os.path.basename(archive_name)}...")
        subprocess.run(["tar", "-xzf", archive_name, "-C", EXTRACT_DIR], check=True)
        open(marker, "w").close()

    # The archives' directory structure is irrelevant (every question refers
    # to an image by basename), so flatten everything into EXTRACT_DIR.
    for root, _dirs, files in os.walk(EXTRACT_DIR):
        if root == EXTRACT_DIR:
            continue
        for f in files:
            if f.lower().endswith(_IMAGE_EXTS):
                os.replace(os.path.join(root, f), os.path.join(EXTRACT_DIR, f))


needed = sorted({os.path.basename(q["Image"]) for q in questions})
to_stage = [
    b for b in needed
    if b not in IMAGE_CACHE and not os.path.exists(os.path.join(EXTRACT_DIR, b))
]
print(f"{len(needed)} images needed, {len(needed) - len(to_stage)} already covered "
      f"(cached or already local), {len(to_stage)} to fetch.")

if to_stage:
    # One directory listing on Drive instead of one existence check per file.
    drive_have = set(os.listdir(DRIVE_FLAT_DIR)) if os.path.isdir(DRIVE_FLAT_DIR) else set()
    if all(b in drive_have for b in to_stage):
        srcs = [os.path.join(DRIVE_FLAT_DIR, b) for b in to_stage]
        dsts = [os.path.join(EXTRACT_DIR, b) for b in to_stage]
        # Modest worker count: this still hits Drive's rate-limited API.
        with ThreadPoolExecutor(max_workers=8) as pool:
            done = 0
            for _ in pool.map(_stage_file, srcs, dsts):
                done += 1
                if done % 100 == 0 or done == len(to_stage):
                    print(f"Staged {done}/{len(to_stage)} images from Drive")
    else:
        print(f"{DRIVE_FLAT_DIR} is missing or incomplete -- falling back to "
              f"copying + extracting the archives locally.")
        _extract_archives_locally()

    still_missing = [b for b in to_stage if not os.path.exists(os.path.join(EXTRACT_DIR, b))]
    if still_missing:
        raise RuntimeError(
            f"{len(still_missing)} needed images are still missing, e.g. {still_missing[:5]}"
        )


2174 images needed, 2174 already covered (cached or already local), 0 to fetch.


In [19]:
#* Load vLLM's compile cache from a single tarball on Drive (if a previous
#* session produced one) so this session's engine startups don't recompile
#* kernels from scratch -- see the VLLM_CACHE_ROOT note in the env-setup cell.
import shutil
import subprocess

os.makedirs(VLLM_CACHE_LOCAL, exist_ok=True)
if os.path.exists(VLLM_CACHE_STORE):
    try:
        subprocess.run(["tar", "-xf", VLLM_CACHE_STORE, "-C", VLLM_CACHE_LOCAL], check=True)
        print(f"Restored vLLM compile cache from {VLLM_CACHE_STORE}")
    except subprocess.CalledProcessError:
        # A cache that fails to extract (corrupt/truncated -- e.g. an old
        # tarball from before the atomic write in sync_vllm_cache_to_drive
        # below existed) should degrade to a cold start, not take the whole
        # notebook down with it: this is a speed optimization, not something
        # correctness depends on, and BREAK_ON_ERR would otherwise disconnect
        # the runtime on cell 1 of every future session using this cache.
        print(f"WARNING: could not extract {VLLM_CACHE_STORE} -- starting "
              f"with a cold compile cache; it will be rebuilt this run.")
else:
    print("No vLLM compile cache on Drive yet -- this run will build and store one.")


def sync_vllm_cache_to_drive() -> None:
    """Tar up the local compile cache and land it on Drive atomically (temp
    name + os.replace, same pattern as _stage_file elsewhere in this notebook) -- writing the tar straight to a Drive path would risk
    leaving a truncated archive if the kernel dies mid-write, and that risk
    is no longer hypothetical now that BREAK_ON_ERR can kill the runtime
    out from under an in-flight write. Called after every model pass, not
    just once at the end, so a crash on a later model (or a BREAK_ON_ERR
    disconnect) doesn't throw away compile work already paid for by the
    models that finished first."""
    vllm_subdir = os.path.join(VLLM_CACHE_LOCAL, "vllm")
    if not os.path.isdir(vllm_subdir):
        return  # nothing compiled yet
    tmp_local = "/content/vllm_cache.tar.tmp"
    subprocess.run(["tar", "-cf", tmp_local, "-C", VLLM_CACHE_LOCAL, "vllm"], check=True)
    tmp_drive = VLLM_CACHE_STORE + ".part"
    shutil.copyfile(tmp_local, tmp_drive)
    os.replace(tmp_drive, VLLM_CACHE_STORE)
    os.remove(tmp_local)

Restored vLLM compile cache from /content/drive/MyDrive/qwen3.5-quant-bench/vllm_cache.tar


In [20]:
import json
import os

RESULTS_DIR = os.path.join(PROJECT_DIR, "test_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Small, git-friendly benchmark artifacts (per-model metrics + eval output,
# see the eval cell below) go in a repo-local `benchmarks/` dir so they get
# committed alongside the notebook (see Commit discipline in CLAUDE.md) --
# unlike RESULTS_DIR above, which holds the raw per-item model outputs
# (Drive-only, not in git; see Repo shape in CLAUDE.md).
BENCHMARK_DIR = os.path.join(PROJECT_DIR, "benchmarks")
os.makedirs(BENCHMARK_DIR, exist_ok=True)

all_metrics = {}
for key, cfg in MODEL_CONFIGS.items():
    if not BENCHMARK_MODELS.get(key):
        print(f"Skipping {key}: disabled in BENCHMARK_MODELS.")
        continue
    if "weights_path" not in cfg:
        print(f"Skipping {key}: weights were not downloaded (missing repo_id?).")
        continue

    print(f"\n=== Running inference: {key} ===")
    results_json = os.path.join(RESULTS_DIR, f"{key}_MME_res.json")
    try:
        run_output = run_inference_pass(
            cfg["weights_path"], results_json, questions, quantization=cfg["quantization"]
        )
        all_metrics[key] = run_output["metrics"]

        with open(os.path.join(BENCHMARK_DIR, f"{key}_metrics.json"), "w") as f:
            json.dump(run_output["metrics"], f, indent=2)
    finally:
        # Persist both the image cache and the vLLM compile cache right
        # after this model's pass, whether it succeeded or raised partway
        # through (a bad request, an engine crash, a BREAK_ON_ERR
        # disconnect about to fire) -- a crash on model N shouldn't throw
        # away cache-filling/compile work already done by models before it,
        # or done so far within model N itself. build_chat (above) fills
        # IMAGE_CACHE as run_inference_pass runs, so it can hold real,
        # worth-keeping progress even from a pass that didn't finish.
        save_image_cache()
        sync_vllm_cache_to_drive()

with open(os.path.join(BENCHMARK_DIR, "benchmark_summary.json"), "w") as f:
    json.dump(all_metrics, f, indent=2)


=== Running inference: qwen3.5-2b ===
INFO 09-22 05:25:37 [api_utils.py:286] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 32768, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 64, 'compilation_config': {'mode': None, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': [], 'ir_enable_torch_wrap': None, 'splitting_ops': None, 'compile_mm_encoder': False, 'cudagraph_mm_encoder': False, 'encoder_cudagraph_token_budgets': [], 'encoder_cudagraph_max_vision_items_per_batch': 0, 'encoder_cudagraph_max_frames_per_batch': None, 'compile_sizes': None, 'compile_ranges_endpoints': None, 'inductor_compile_config': {'enable_auto_functionalized_v2': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': None, 'cudagraph_num_of_warmups': 0, 'cudagraph_capture_sizes': [1, 2, 4, 8, 16, 32, 64], 'cudagraph_copy_inputs': F

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


WARNING 09-22 05:25:59 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 09-22 05:28:49 [hf.py:547] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.


[transformers] Qwen3VL video processing does not apply the per-frame pixel cap the reference implementation (qwen-vl-utils) applies, so some videos cost far more tokens than they would there. In v5.22 the capped behavior will become the default and `cap_pixels_per_frame` will be removed. Pass `cap_pixels_per_frame=True` to adopt the reference behavior now, or `False` to keep the current behavior and silence this warning.


INFO 09-22 05:29:18 [base.py:244] Multi-modal warmup completed in 28.310s
INFO 09-22 05:29:20 [base.py:244] Readonly multi-modal warmup completed in 2.101s


...

Processed prompts:  36%|███▌      | 20/56 [00:09<00:21,  1.68it/s, est. speed input: 14312.65 toks/s, output: 40.46 toks/s]

INFO 09-22 06:33:21 [loggers.py:310] Engine 000: Avg prompt throughput: 23426.1 tokens/s, Avg generation throughput: 104.2 tokens/s, Running: 35 reqs, Waiting: 0 reqs, GPU KV cache usage: 7.9%, Prefix cache hit rate: 0.0%, MM cache hit rate: 0.4%


Processed prompts: 100%|██████████| 56/56 [00:12<00:00,  4.31it/s, est. speed input: 19894.93 toks/s, output: 531.88 toks/s]

Processed 2360/2360
INFO 09-22 06:33:24 [utils.py:620] [shutdown] Process manager: send sigterm to process EngineCore


WARNING 09-22 06:33:29 [utils.py:640] [shutdown] Process manager: force killing remaining processes count=1
WARNING 09-22 06:33:29 [utils.py:645] [shutdown] Process manager: force killing remaining process EngineCore pid 12675
Saved 2174 cached images to /content/drive/MyDrive/qwen3.5-quant-bench/base64_image_cache.json


In [21]:
import subprocess

# cloned git repository, not dataset cache
EVAL_SCRIPT = "/content/MME-RealWorld/evaluation/eval_your_results.py"

for key in all_metrics:  # only models that actually ran inference above
    results_json = os.path.join(RESULTS_DIR, f"{key}_MME_res.json")
    eval_txt = os.path.join(BENCHMARK_DIR, f"{key}_output.txt")
    print(f"Evaluating {key}...")
    with open(eval_txt, "w") as f:
        subprocess.run(
            ["python", EVAL_SCRIPT, "--results_file", results_json],
            stdout=f, stderr=subprocess.STDOUT, check=True,
        )

Evaluating qwen3.5-2b...
Evaluating qwen3.5-2b-awq...


In [ ]:
def _fmt(x, prec=3):
    return "n/a" if x is None else f"{x:.{prec}f}"

header = (
    f"{'model':<20}{'lat avg':>10}{'lat p99':>10}{'ttft avg':>10}"
    f"{'tpot avg':>10}{'tok/s':>10}{'peak GB':>10}"
)
print(header)
print("-" * len(header))
for key, m in all_metrics.items():
    print(
        f"{key:<20}"
        f"{_fmt(m['latency']['avg']):>10}"
        f"{_fmt(m['latency']['p99']):>10}"
        f"{_fmt(m['ttft']['avg']):>10}"
        f"{_fmt(m['tpot']['avg']):>10}"
        f"{_fmt(m['throughput']['output_tokens_per_second'], 1):>10}"
        f"{_fmt(m['memory_gb']['peak_allocated'], 2):>10}"
    )

print(f"\nFull per-run metrics (incl. p50/p90/p99, time-weighted latency, queue/prefill/decode "
      f"breakdown, preemptions, unload residual): {BENCHMARK_DIR}/benchmark_summary.json")
print(f"Accuracy: {BENCHMARK_DIR}/<model>_output.txt")

model                  lat avg   lat p99  ttft avg  tpot avg     tok/s   peak GB
--------------------------------------------------------------------------------
qwen3.5-2b              16.583    51.229     5.260     0.434     155.5     42.37
qwen3.5-2b-awq          19.464    54.167     5.770     0.408     156.4     42.51

Full per-run metrics (incl. p50/p90/p99, time-weighted latency, queue/prefill/decode breakdown, preemptions, unload residual): /content/drive/MyDrive/qwen3.5-quant-bench/benchmarks/benchmark_summary.json
Accuracy: /content/drive/MyDrive/qwen3.5-quant-bench/benchmarks/<model>_output.txt


In [23]:
from google.colab import runtime

runtime.unassign()
